In [4]:
import json
import os
from pathlib import Path
import urllib.request

import openai


def load_env_file(path: str = ".env"):
    env_path = Path(path)
    if not env_path.exists():
        return

    for line in env_path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue

        key, value = line.split("=", 1)
        os.environ.setdefault(key.strip(), value.strip().strip('"').strip("'"))


load_env_file()

MOVIE_API_BASE_URL = os.getenv(
    "MOVIE_API_BASE_URL", "https://nomad-movies.nomadcoders.workers.dev"
)

client = openai.OpenAI()
messages = []

In [5]:
def _fetch_movie_api(path: str):
    url = f"{MOVIE_API_BASE_URL}{path}"
    request = urllib.request.Request(
        url,
        headers={
            "Accept": "application/json",
            "User-Agent": "Mozilla/5.0",
        },
    )
    with urllib.request.urlopen(request, timeout=10) as response:
        return json.loads(response.read().decode("utf-8"))


def get_popular_movies():
    movies = _fetch_movie_api("/movies")
    return json.dumps(movies)


def get_movie_details(id):
    details = _fetch_movie_api(f"/movies/{id}")
    return json.dumps(details)


def get_movie_credits(id):
    credits = _fetch_movie_api(f"/movies/{id}/credits")
    return json.dumps(credits)


FUNCTION_MAP = {
    "get_popular_movies": get_popular_movies,
    "get_movie_details": get_movie_details,
    "get_movie_credits": get_movie_credits,
}

In [6]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_popular_movies", # function: get_popular_movies()
            "description": "Get the most popular movie.",
            "parameters": {
                "type": "object",
                "properties": {},
                "required": [],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_details", # function: get_movie_details(id)
            "description": "Get details for a movie by its id.",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {
                        "type": "string",
                        "description": "The movie id.",
                    }
                },
                "required": ["id"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_credits", # function: credits(id)
            "description": "Get credit score for a movie by its id.",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {
                        "type": "string",
                        "description": "The movie id.",
                    }
                },
                "required": ["id"],
            },
        },
    },
]

In [10]:
from openai.types.chat import ChatCompletionMessage


def process_ai_response(message: ChatCompletionMessage):

    if message.tool_calls:
        messages.append(
            {
                "role": "assistant",
                "content": message.content or "",
                "tool_calls": [
                    {
                        "id": tool_call.id,
                        "type": "function",
                        "function": {
                            "name": tool_call.function.name,
                            "arguments": tool_call.function.arguments,
                        },
                    }
                    for tool_call in message.tool_calls
                ],
            }
        )

        for tool_call in message.tool_calls:
            function_name = tool_call.function.name
            arguments = tool_call.function.arguments

            # print(f"Calling function: {function_name} with {arguments}")

            try:
                arguments = json.loads(arguments)
            except json.JSONDecodeError:
                arguments = {}

            function_to_run = FUNCTION_MAP.get(function_name)

            if function_to_run is None:
                result = f"Error: unknown function '{function_name}'."
                print(result)
            else:
                try:
                    result = function_to_run(**arguments)
                except Exception as e:
                    result = f"Error running {function_name}: {e}"
                    print(result)

            # print(f"Ran {function_name} with args {arguments} for a result of {result}")

            messages.append(
                {
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "name": function_name,
                    "content": str(result),
                }
            )

        call_ai()
    else:
        messages.append({"role": "assistant", "content": message.content})
        print(f"AI: {message.content}")


def call_ai():
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        tools=TOOLS,
    )
    process_ai_response(response.choices[0].message)

In [13]:
while True:
    message = input("Send a message to the LLM...")
    if message == "quit" or message == "q":
        break
    else:
        messages.append({"role": "user", "content": message})
        print(f"User: {message}")
        call_ai()

User: tell me the most popular movie right now.
AI: The most popular movie right now is **"Your Heart Will Be Broken"**.

- **Release Date:** March 26, 2026
- **Overview:** High school student Polina is saved from bullying at her new school and makes a deal with the main bully Bars: he must pretend to be her boyfriend and protect her, and she must do everything he says. During this game, the couple develops real feelings, but her family and classmates have reasons to separate the lovers.
- **Vote Average:** 6.531
- **Vote Count:** 48
- **Popularity:** 1190.699

![Your Heart Will Be Broken](https://image.tmdb.org/t/p/w780/iGpMm603GUKH2SiXB2S5m4sZ17t.jpg)

If you would like more details or information about anything else, just let me know!
User: what is the movie that has movie id 550?
AI: The movie with ID **550** is **"Fight Club."**

- **Release Date:** October 15, 1999
- **Genres:** Drama, Thriller
- **Overview:** A ticking-time-bomb insomniac and a slippery soap salesman channel pri

In [14]:
messages

[{'role': 'user', 'content': 'hello, tell me the most popular movie.'},
 {'role': 'assistant',
  'content': '',
  'tool_calls': [{'id': 'call_w8ZzXTtePXQ5mFF9Tbaqsq9c',
    'type': 'function',
    'function': {'name': 'get_popular_movies', 'arguments': '{}'}}]},
 {'role': 'tool',
  'tool_call_id': 'call_w8ZzXTtePXQ5mFF9Tbaqsq9c',
  'name': 'get_popular_movies',
  'content': '[{"adult": false, "backdrop_path": "https://image.tmdb.org/t/p/w1280/1x9e0qWonw634NhIsRdvnneeqvN.jpg", "genre_ids": [10749, 18], "id": 1523145, "original_language": "ru", "original_title": "\\u0422\\u0432\\u043e\\u0451 \\u0441\\u0435\\u0440\\u0434\\u0446\\u0435 \\u0431\\u0443\\u0434\\u0435\\u0442 \\u0440\\u0430\\u0437\\u0431\\u0438\\u0442\\u043e", "overview": "High school student Polina is saved from bullying at her new school and makes a deal with the main bully Bars: he must pretend to be her boyfriend and protect her, and she must do everything he says. During this game, the couple develops real feelings, but he